# RIXS–QFI paper figure reproduction

This notebook generates the numerical figures for the paper, covering the Ce and Ta systems. Figure sections follow the numbering in the paper; paired main and supplementary figures are kept together.

Run the notebook from beginning to end. Shared preparation precedes the figure sections because later calculations reuse earlier results. The configuration and working variables are subsequently replaced when switching from Ce to Ta.

The notebook uses precomputed QFI and optimization results; it does not repeat the original fitting or QFI-maximization searches. Supplementary Fig. S4 preparation is performed inline from saved optimization results.

RIXS denotes resonant inelastic X-ray scattering. QFI denotes quantum Fisher information. Here, normalized QFI (nQFI) is the QFI divided by the model-derived maximal one-partite QFI. The red line at nQFI = 1 marks the two-partite entanglement threshold under the reference-state assumptions described in the paper.

## Python environment

Required packages: NumPy, Matplotlib, SciPy, NetworkX, and edrixs.

The analysis modules `plot2`, `optEDRIXS`, `utils_Ce`, `utils_Ta`, and `witness_utils` must be importable. Place them in the notebook directory, install them, or add their directory to the Python import path. Setting `RIXS_QFI_DATA_ROOT` specifies the data directory; it does not configure the module import path.

## Required input files

Place the following files and scan directories in the working directory, or in the directory specified by `RIXS_QFI_DATA_ROOT`.

### Configuration and plotting

- `config_Ce_t2g.py`
- `config_Ta_Eg.py`
- `white_turbo.npy`

### Ce parameter sets and precomputed results

- `Ba3CeIr2O9_test4_collectedV4V.txt`
- `Ce_t2g_momdep_outputsV4Vr.npz`
- `Ce_t2g_momdep_pol_outputsV4Vr.npz`
- `Ce_t2g_Edep_outputsV4Vr.npz`
- `Ce_t2g_Edep_pol_outputsV4Vr.npz`
- `Ce_curves2.npz`
- `Ce_t2g_maxwal_P.npz`
- `Ce_t2g_maxwal_M.npz`

### Ta parameter sets and precomputed results

- `Ba3TaIr2O9_collectedV.txt`
- `Ta_Eg_momdep_outputsV.npz`
- `Ta_Eg_momdep_pol_outputsV.npz`
- `Ta_Eg_Edep_outputsV.npz`
- `Ta_Eg_Edep_pol_outputsV.npz`

### Experimental scan files

Each scan must be available as `scan_<number>.npz` inside the corresponding directory.

| Directory | Required scan numbers |
|---|---|
| `data_Ba3CeIr2O9/` | 114, 115, 118, 119, 121, 134, 135, 136, 137, 138, 140, 141, 147, 148, 149, 150, 151, 152, 153, 154, 155, 157, 160, 161, 162, 164 |
| `data_Ba3TaIr2O9/` | 3, 6, 9, 12, 15, 18, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 41, 45, 48, 49 |

This checklist covers files referenced directly by the notebook. The configuration files and analysis modules may require additional files; those dependencies must also be available.

## Generated figures

Figures are saved as PDF files in the data directory. Plot outputs are intentionally not stored in this notebook.

- Main: `Ce_EnergyMap_new.pdf`, `Ce_LMap_new.pdf`, `Ce_CutMap_new.pdf`, `level_new.pdf`, `Ce_exp_new.pdf`, `Ce_simulation_new.pdf`
- Supplement: `Ce_exp_new_nonorm.pdf`, `Ce_simulation_new_nonorm.pdf`, `Ce_QFI_evenvsodd_ratio.pdf`, `Ta_EnergyMap_new.pdf`, `Ta_LMap_new.pdf`, `Ta_CutMap_new.pdf`, `Ta_exp_new.pdf`, `Ta_simulation_new.pdf`

## Reading the calculations

Scientific context is provided by the main manuscript (Methods, Table I, and figure captions) and Supplementary Information (Secs. III–VI and Table S1). The notes below explain how that context maps to the arrays and selections used here. Numerical or plotting conventions whose rationale is not specified in those sources are identified as implementation choices.


### Matplotlib layout state

For reproducibility, every saved figure explicitly restores the Matplotlib
state that was effective when the corresponding production figure was made.
This is deliberate: in the original working notebooks some of those settings
were inherited from earlier diagnostic cells.  The diagnostic plots are not
included here, but their plotting-state side effects are reproduced directly.
Slight differences in layout can occur as different versions of matplotlib
were used for different plots (3.8.0, 3.9.4, 3.11.1, 3.11.2)


The setup selects the data directory, configures shared plotting helpers, and defines the custom colormap and QFI figure style. Energy-loss quantities used with the thermal factor are in eV, while incident-energy plot axes are in keV.


In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.signal import savgol_filter

import edrixs
from plot2 import plot_image, make_theme, plot_label, format_cbar, enlarge
from optEDRIXS import import_config
from utils_Ce import RIXS_runner as RIXS_runner_Ce
from utils_Ta import RIXS_runner as RIXS_runner_Ta
from witness_utils import SparseBinary, get_fock_bin_by_N_S

DATA_ROOT = Path(os.environ.get('RIXS_QFI_DATA_ROOT', '.')).expanduser().resolve()
os.chdir(DATA_ROOT)
def turbo_w():
    return ListedColormap(np.load(DATA_ROOT / 'white_turbo.npy'))



# ------------------------------------------------------------------
# Matplotlib states used by the published figures
# ------------------------------------------------------------------
# These reproduce the effective rcParams of the original production
# notebooks, but are applied explicitly before every saved paper figure.
# This prevents figure layout from depending on execution order or on
# diagnostic figures that existed in the working notebooks.

def paper_rc_default9():
    """State used by Ce Figs. 2–5."""
    plt.rcdefaults()
    plt.rcParams['font.size'] = 9

def paper_rc_qfi9():
    """State inherited by the Ce/Ta QFI and Ta map/cut figures."""
    plt.rcdefaults()
    plt.rcParams.update({
        'figure.figsize': (3.4, 2.8),
        'figure.dpi': 300,
        'font.size': 9,
        'axes.labelsize': 9,
        'axes.titlesize': 9,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'legend.frameon': False,
        'axes.linewidth': 0.8,
    })

def paper_rc_s4():
    """State used by Supplementary Fig. S4."""
    plt.rcdefaults()

def paper_rc_level7():
    """Exact default state used by the published Ce Fig. 5."""
    plt.rcdefaults()
    plt.rcParams['font.size'] = 7


## Ce — shared preparation for Figs. 2–7 and S2–S4

The Ce configuration supplies the model, reference spectra, geometry, and fitted-parameter names. Each row of `coordsfin` is a fitted parameter set, with columns ordered according to `cfg.greedy_bounds`. Model intensities are evaluated for every row to obtain separate overall scale factors from the momentum-dependent and incident-energy-dependent reference data.

For Ce, row **0** has the smallest overall fitting distance and supplies the representative theoretical curves. The QFI files contain precomputed results. The temperatures used here are 9 K for Ce and, below, 11 K for Ta (main manuscript, Methods).

The momentum conversion uses the Ce dimer separation `d = 2.53614` Å and the reciprocal-lattice factor `0.426991866` Å⁻¹. `Ls` is the dimensionless reciprocal-lattice coordinate; multiplying by `Lfactor` expresses momentum in units of 2π/d. The incident-energy conversion follows `(omega + 11774.5) / 1000`.


In [ ]:
cfg = import_config('config_Ce_t2g.py')

coordsfin = np.loadtxt('Ba3CeIr2O9_test4_collectedV4V.txt')
labels0 = list(cfg.greedy_bounds.keys())

cmap, cmapE = ([], [])

for ind in range(len(coordsfin)):
    dp = {labels0[i]: coordsfin[ind, i] for i in range(len(labels0))}
    dp2 = {**dp, **cfg.fixed_params_greedy}
    cmap.append(cfg.rixs_funV(dp2))
    cmapE.append(cfg.rixs_funV_E(dp2))

scale_factor = np.array([np.sum(cfg.rixs_ref) / np.sum(cmap[q]) for q in range(len(cmap))])
scale_factor_E = np.array([np.sum(cfg.rixs_ref_Emap) / np.sum(cmapE[q]) for q in range(len(cmapE))])

d = 2.53614
Lfactor = 0.426991866 * d / (2 * np.pi)
Ls = np.arange(0.0, 20.1, 0.1)
newmomenta = Ls * Lfactor
L = 18.94
T = 9.0
kB = 8.6173303e-05
beta = 1 / (kB * T)

mom = np.load('Ce_t2g_momdep_outputsV4Vr.npz')
theorSpectra = np.vstack([mom['theorSpectra']])
numNegTheor = np.vstack([mom['numNegTheor']])
denomTheor = np.vstack([mom['denomTheor']])

mom_pol = np.load('Ce_t2g_momdep_pol_outputsV4Vr.npz')
theorSpectraM0 = mom_pol['theorSpectraM0']
theorSpectraMpi2 = mom_pol['theorSpectraMpi2']
nominator1M = mom_pol['nominator1']
nominator2M = mom_pol['nominator2']

omega = np.linspace(-10, 10, 201) - 558.5
omega11 = (omega + 11774.5) / 1000

edep = np.load('Ce_t2g_Edep_outputsV4Vr.npz')
theorSpectraE = edep['theorSpectra']
numNegTheorE = edep['numNegTheor']
denomTheorE = edep['denomTheor']

edep_pol = np.load('Ce_t2g_Edep_pol_outputsV4Vr.npz')
theorSpectraE0 = edep_pol['theorSpectraE0']
theorSpectraEpi2 = edep_pol['theorSpectraEpi2']
nominator1E = edep_pol['nominator1']
nominator2E = edep_pol['nominator2']


### Experimental QFI preparation (Figs. 6–7)


**Momentum-dependent experimental nQFI (Fig. 6).** Scans 147–155, 157, 160–162, and 164 are loaded in the order used for the measured momentum series. Their model-grid positions are matched to the configured `Ls` values after rounding to one decimal place.

For every spectrum, the code subtracts a constant background estimated over `250 ≤ scan['x'] ≤ 350` in the stored scan coordinate, applies the self-absorption correction, and integrates over `0.2 < energy loss < 2.0` eV. The Ce analysis window excludes the low-energy elastic region and covers the modeled intra-t₂g excitations (Supplementary Information, Sec. V). The exact background interval is an implementation choice; the scan energy offset is applied separately.

The indices **`[0, 3, 4, 11, 14, 18]`** select the six fitted parameter sets retained after filtering nonphysical solutions, including `F4 > F2` or saturation of fitting bounds. For each retained set, the measured positive-energy contribution is combined with the model negative-energy contribution and normalized by the corresponding one-partite maximum. Both intensity-normalization estimates are used, giving an equally weighted ensemble of **12** estimates. Its mean and standard deviation provide the plotted points and error bars (Supplementary Information, Sec. V); these error bars describe the fit/normalization ensemble spread.

The integral uses a rectangular sum and rounds the energy spacing to two decimal places. This numerical convention is retained from the supplied calculation.


In [ ]:
atab = np.round(cfg.geometry['Ls'], 1)
indices = np.nonzero(np.isin(np.round(np.arange(0.0, 20.1, 0.1), 1), atab))[0]
Mmap_data = dict(eloss=[], intensity=[])

for scan_no in np.hstack([np.arange(147, 156), 157, np.arange(160, 163), 164]):
    scan = dict(np.load(f'data_Ba3CeIr2O9/scan_{scan_no}.npz'))
    eloss = (scan['x'] - scan['xoffset']) / 1000
    choose = (eloss > 0.2) & (eloss < 2.0)
    Mmap_data['eloss'].append(eloss[choose])
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Mmap_data['intensity'].append(((scan['y'] - bg) / scan['ab_factor'])[choose])

QFI_data_L2, QFI_data_L2E = ([], [])

for q in [0, 3, 4, 11, 14, 18]:
    pos = []
    for i in range(len(Mmap_data['intensity'])):
        e = np.asarray(Mmap_data['eloss'][i])
        y = np.asarray(Mmap_data['intensity'][i])
        fac = 2 * np.tanh(beta * e / 2) * (1 - np.exp(-beta * e))
        step = np.round(e[1] - e[0], 2)
        pos.append(np.sum(y * fac * step))
    choose0 = indices
    pos = np.asarray(pos)
    neg = numNegTheor[q][choose0]
    num_m = pos + neg * scale_factor[q]
    num_e = pos + neg * scale_factor_E[q]
    QFI_data_L2.append(num_m / (denomTheor[q][choose0] * scale_factor[q]))
    QFI_data_L2E.append(num_e / (denomTheor[q][choose0] * scale_factor_E[q]))

QFI_data = np.vstack([QFI_data_L2, QFI_data_L2E])
QFI_mean = np.mean(QFI_data, axis=0)
QFI_std = np.std(QFI_data, axis=0)


**Incident-energy-dependent experimental nQFI (Fig. 7).** Scans 121, 134–138, 140, and 141 supply the measured map and incident energies. Spectra are interpolated onto the first scan's energy grid after energy-offset, background, and self-absorption corrections.

The QFI integral itself uses `cfg.rixs_ref_Emap` and `cfg.resolution['eloss_Emap']`, rather than the locally assembled map. These configuration-provided arrays therefore determine the integration data and window. The same six retained Ce fits and two normalizations produce a 12-member ensemble.

Incident energies are matched to the precomputed grid by an exact dictionary lookup after the stored energy conversion. 


In [ ]:
Emap_data = dict(eloss=None, omega=[], intensity=[])

for scan_no in [121, 134, 135, 136, 137, 138, 140, 141]:
    scan = dict(np.load(f'data_Ba3CeIr2O9/scan_{scan_no}.npz'))
    if Emap_data['eloss'] is None:
        Emap_data['eloss'] = scan['x']
    Emap_data['omega'].append(scan['Ei'])
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Emap_data['intensity'].append(np.interp(
        Emap_data['eloss'],
        scan['x'] - scan['xoffset'],
        (scan['y'] - bg) / scan['ab_factor'],
    ))

Emap_data['intensity'] = np.vstack(Emap_data['intensity']).T
Emap_data['eloss'] = Emap_data['eloss'] * 0.001
omega_data = np.array(Emap_data['omega'])
omega_select = omega_data * 1000 - 11774.5

QFI_data_Edep, QFI_data_Edep_E = ([], [])

for q in [0, 3, 4, 11, 14, 18]:
    pos = []
    for i in range(len(cfg.rixs_ref_Emap[0])):
        y = np.asarray(cfg.rixs_ref_Emap[:, i])
        e = cfg.resolution['eloss_Emap']
        fac = 2 * np.tanh(beta * e / 2) * (1 - np.exp(-beta * e))
        step = np.round(e[1] - e[0], 2)
        pos.append(np.sum(y * fac * step))
    pos_in_B = {val: idx for idx, val in enumerate(omega)}
    choose2 = np.array([pos_in_B[a] for a in omega_select])
    pos = np.asarray(pos)
    neg = numNegTheorE[q][choose2]
    num_m = pos + neg * scale_factor[q]
    num_e = pos + neg * scale_factor_E[q]
    QFI_data_Edep.append(num_m / (denomTheorE[q][choose2] * scale_factor[q]))
    QFI_data_Edep_E.append(num_e / (denomTheorE[q][choose2] * scale_factor_E[q]))

QFI_data_E = np.vstack([QFI_data_Edep, QFI_data_Edep_E])
QFI_mean_E = np.mean(QFI_data_E, axis=0)
QFI_std_E = np.std(QFI_data_E, axis=0)


### RIXS maps and cuts: model preparation (Figs. 2–4)


**Momentum-map preparation (Fig. 3).** The measured scans are interpolated onto a common energy grid. A denser simulation is then evaluated for `L = 6.0–20.0`, using the experimental geometry conversion and the representative Ce fit, row **0**. `dpbest` contains its fitted parameters; `dp2best` adds parameters held fixed during fitting.

The numerical geometry constants specify the reciprocal-lattice and wavelength conversions used by this calculation. The configuration is updated for this map before the subsequent cut and energy-map calculations.


In [ ]:
Ldep_data = dict(eloss=None, HKL=[], intensity=[])

for scan_no in np.hstack([np.arange(147, 156), 157, np.arange(160, 163), 164]):
    scan = dict(np.load(f'data_Ba3CeIr2O9/scan_{scan_no}.npz'))
    if Ldep_data['eloss'] is None:
        Ldep_data['eloss'] = scan['x']
    Ldep_data['HKL'].append(scan['HKL'])
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Ldep_data['intensity'].append(np.interp(
        Ldep_data['eloss'],
        scan['x'] - scan['xoffset'],
        (scan['y'] - bg) / scan['ab_factor'],
    ))

Ldep_data['intensity'] = np.vstack(Ldep_data['intensity']).T
Ldep_data['HKL'] = np.vstack(Ldep_data['HKL'])
Ldep_data['eloss'] = Ldep_data['eloss'] * 0.001
Ls0 = np.arange(6.0, 20.1, 0.1)
Qs0 = np.sqrt(1.228965 ** 2 / 4 + (Ls0 * 0.426991866) ** 2)
tths0 = np.rad2deg(np.arcsin(1.1054837271511369 * Qs0 / 4 / np.pi) * 2)
cfg.resolution['eloss'] = cfg.eloss0
cfg.geometry['Ls'] = Ls0
cfg.geometry['Qs'] = Qs0
cfg.geometry['tths'] = tths0
cfg.geometry['thins'] = tths0 / 2 - np.rad2deg(np.arctan(1.228965 / 2 / Ls0 / 0.426991866))
dpbest = {labels0[i]: coordsfin[0, i] for i in range(len(labels0))}
dp2best = {**dpbest, **cfg.fixed_params_greedy}
test = cfg.rixs_funV(dp2best)


**Representative-cut preparation (Fig. 4).** Scans `[115, 114, 161, 155, 119, 118]` correspond to the six cuts compared with the model. Simulations use two L values, `17.48` and `14.57`, for each H value `−3.3`, `−0.5`, and `0.5`. The explicit scattering/incidence-angle arrays specify those measurement geometries.

The order of the six scans must agree with the order of the six columns assembled in `Cmap_calc`. Row **0** supplies the model parameters and momentum-map intensity normalization. The runner uses the fitted `Gam_c`, the configured broadening `sigma`, and `fraction=0`.


In [ ]:
Cmap_data = []

for scan_no in [115, 114, 161, 155, 119, 118]:
    scan = dict(np.load(f'data_Ba3CeIr2O9/scan_{scan_no}.npz'))
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Cmap_data.append(dict(
        x=(scan['x'] - scan['xoffset']) / 1000.0,
        y=(scan['y'] - bg) / scan['ab_factor'],
        yerr=scan['yerr'] / scan['ab_factor'],
        HKLc=[scan['HKL'][0], scan['HKL'][1], scan['HKL'][2], scan['qc']],
    ))

Ls1 = np.array([17.48, 14.57])
Qs1 = np.sqrt(1.228965 ** 2 / 4 + (Ls1 * 0.426991866) ** 2)
tths1 = np.array([[96.6, 81.5], [82.3, 66.6], [82.3, 66.6]])
thins1 = np.array([[19.75, 7.62], [36.43, 27.67], [45.86, 38.96]])
test1 = []
cfg.geometry['Ls'] = Ls1
cfg.geometry['Qs'] = Qs1
cfg.resolution['eloss'] = np.arange(-0.3975, 2.5025, 0.01)

for ind, hh in enumerate([-3.3, -0.5, 0.5]):
    cfg.geometry['tths'] = tths1[ind]
    cfg.geometry['HKL'] = [hh, 0.0, 0.0]
    cfg.geometry['thins'] = thins1[ind]
    pars = dict(
        Gam_c=dp2best['Gam_c'],
        sigma=cfg.true_values['sigma'],
        fraction=0.0,
        geometry=cfg.geometry,
        resolution=cfg.resolution,
        T=cfg.temperature,
        eline=None,
    )
    runner = RIXS_runner_Ce(
        pars,
        [
            cfg.sptab2i,
            cfg.sptab2n1,
            cfg.sptab2n2,
            cfg.sptab4i,
            cfg.sptab4n1,
            cfg.sptab4n2,
            cfg.sptabtr,
        ],
    )
    test1.append(runner(dp2best))

test2 = np.hstack(test1)
scale_factorT = scale_factor[0]
Cmap_calc = [
    dict(x=cfg.resolution['eloss'], y=test2[:, 0] * scale_factorT, Q='(-3.3, 0, 17.48)', qc=6),
    dict(x=cfg.resolution['eloss'], y=test2[:, 1] * scale_factorT, Q='(-3.3, 0, 14.57)', qc=5),
    dict(x=cfg.resolution['eloss'], y=test2[:, 2] * scale_factorT, Q='(-0.5, 0, 17.48)', qc=6),
    dict(x=cfg.resolution['eloss'], y=test2[:, 3] * scale_factorT, Q='(-0.5, 0, 14.57)', qc=5),
    dict(x=cfg.resolution['eloss'], y=test2[:, 4] * scale_factorT, Q='(0.5, 0, 17.48)', qc=6),
    dict(x=cfg.resolution['eloss'], y=test2[:, 5] * scale_factorT, Q='(0.5, 0, 14.57)', qc=5),
]


**Incident-energy-map preparation (Fig. 2).** The Ce model is evaluated on incident-energy and energy-loss grids using the representative fit. 

Copies of the simulation grids are retained for Fig. 2 so subsequent configuration updates do not change its plotting coordinates.


In [ ]:
cfg.geometry['omega_Emap'] = np.arange(-562.5, -554.5, 0.2)
cfg.resolution['eloss_Emap'] = np.arange(-0.4, 2.01, 0.01)
pars = dict(
    Gam_c=cfg.true_values['Gam_c'],
    sigma=cfg.true_values['sigma'],
    fraction=0.0,
    geometry=cfg.geometry,
    resolution=cfg.resolution,
    T=cfg.temperature,
    eline=None,
)
runner_E = RIXS_runner_Ce(
    pars,
    [cfg.sptab2i, cfg.sptab2n1, cfg.sptab2n2, cfg.sptab4i, cfg.sptab4n1, cfg.sptab4n2, cfg.sptabtr],
    mode='E',
)
test_Edep = runner_E(dp2best)

ce_incident_energy_grid = cfg.geometry['omega_Emap'].copy()
ce_energy_loss_grid = cfg.resolution['eloss_Emap'].copy()


### Level evolution: calculation and plotting functions (Fig. 5)


**Level-evolution path (Fig. 5).** Starting from the representative Ce fit, the calculation first sets SOC and hopping to zero and uses the ideal octahedral angle. It then introduces SOC, trigonal distortion, and hopping in three consecutive 30-point segments. This is the physical sequence illustrated in the main manuscript's Fig. 5.

The endpoints correspond to the fitted SOC, angle, and hopping values reported, to rounded precision, in Table I. `U_dd_param=3.0` and `U_q_param=2.5` are fixed in this path. The calculation sorts the eigenenergies independently at every point; continuous branches are assigned in the next step.


In [ ]:
all_param_labels = [
    'F2_dd',
    'F2_dp',
    'F4_dd',
    'G1_dp',
    'G3_dp',
    'Vdd_delta',
    'Vdd_pi',
    'Vdd_sigma',
    'soc',
    'soc_c',
    'tenDq',
    'theta',
    'a0',
    'kappa',
    'Gam_c',
    'sigma',
    'U_dd_param',
    'U_q_param',
]

def make_base_dp2():
    p = {k: 0.0 for k in all_param_labels}
    p.update({labels0[i]: coordsfin[0, i] for i in range(len(labels0))})
    p.update(cfg.fixed_params_greedy)
    p['soc'] = 0.0
    p['theta'] = np.arccos(1 / np.sqrt(3)) * 180 / np.pi
    p['Vdd_delta'] = p['Vdd_sigma'] = p['Vdd_pi'] = 0.0
    p['U_dd_param'] = 3.0
    p['U_q_param'] = 2.5
    return p

def scan_segment(base, labels, start, end, n):
    out = []
    for a in np.linspace(0, 1, n):
        p = base.copy()
        vals = (1 - a) * np.asarray(start, float) + a * np.asarray(end, float)
        for lab, val in zip(labels, vals):
            p[lab] = float(val)
        out.append(p)
    for lab, val in zip(labels, end):
        base[lab] = float(val)
    return (out, base)

def build_path():
    base = make_base_dp2()
    path = []
    seg, base = scan_segment(base, ['soc'], [0.0], [0.405815063], 30)
    path += seg
    th0 = np.arccos(1 / np.sqrt(3)) * 180 / np.pi
    seg, base = scan_segment(base, ['theta'], [th0], [50.86], 30)
    path += seg
    seg, base = scan_segment(
        base,
        ['Vdd_sigma', 'Vdd_pi', 'Vdd_delta'],
        [0.0, 0.0, 0.0],
        [-1.34318671, -0.00396863835, 0.383107193],
        30,
    )
    path += seg
    return path
dp_path = build_path()
eigvals_all0 = np.vstack([np.sort(np.real(cfg.rixs_funV(p, ed_only=True))) for p in dp_path])


**Tracking level branches.** The lowest **36** levels are followed through the final, hopping-dependent segment, after subtracting the ground-state energy at each point. A two-pass minimum-cost-flow algorithm links levels using energy separation and a velocity-consistency penalty. It tracks energy continuity rather than eigenvector overlap.

The selected settings are `k=8`, `alpha=5.0`, `beta=3.0`, and integer cost scale `1000`. These are numerical tracking choices; the manuscript does not specify a selection criterion for them.


In [ ]:
# Fig. 5 branch provenance
# ------------------------
# The historical min-cost-flow assignment was not deterministic across software
# environments because several branch connections are degenerate/tied. The
# exact branch trajectories actually drawn in the published vector PDF were
# therefore frozen once into Ce_Fig5_published_branches.npz.
#

fig5_branches = np.load("Ce_Fig5_published_branches.npz")


**Plotting functions for Fig. 5.** The 36 levels are grouped as **4 + 16 + 16**, corresponding to the two-site combinations of the single-ion J=1/2 and J=3/2 multiplets discussed in the figure caption. Colors and further index subdivisions encode that level grouping.

Eigenenergies are rounded to six decimal places, and hopping curves are smoothed with a Savitzky–Golay filter (`window_length=5`, `polyorder=1`). 

The RIXS comparison panel uses cuts **2 and 3** of `Cmap_calc`, corresponding to H=−0.5 at the two selected L values.


In [ ]:
# Deterministic branch data for the published Fig. 5.
fig5_x = fig5_branches["hopping_x"]
fig5_y = fig5_branches["hopping_y_smooth"]
fig5_end_energy = fig5_branches["hopping_end_energy"]
fig5_raw_hopping = fig5_branches["raw_hopping_indices"].astype(int)
fig5_raw_level = fig5_branches["raw_level_indices"].astype(int)
fig5_corrected = fig5_branches["corrected_indices"].astype(int)

ctab = ['C3'] * 4 + ['C4'] * 16 + ['C0'] * 16
typetab = [0] * 4 + [1] * 16 + [2] * 16

def plot_SOC(ax):
    soc = np.linspace(0, 0.4, 30)
    for i in range(36):
        evals = eigvals_all0[:30, i] - eigvals_all0[:30, 0]
        evals = np.round(evals, 6)
        ax.plot(soc, evals, color=ctab[i])
    make_theme(ax, ticklength=4)
    ax.set_title('SOC', fontsize=7, pad=3)
    ax.set_xlabel('$\\zeta$ (eV)')
    ax.set_xlim(0, 0.4)
    ax.set_xticks(np.arange(0.0, 0.331, 0.2))
    ax.set_ylabel('Energy loss (eV)')
    ax.set_ylim(-0.2, 2.25)
    ax.set_yticks(np.arange(0.0, 2.1, 0.5))

def plot_CEF(ax):
    dth = np.linspace(0, np.arccos(1.0 / np.sqrt(3.0)) * 180 / np.pi - 50.86, 30)
    for i in range(36):
        evals = eigvals_all0[30:60, i] - eigvals_all0[30:60, 0]
        evals = np.round(evals, 6)
        ax.plot(dth, evals, color=ctab[i])
    make_theme(ax, ticklength=4)
    ax.tick_params(labelleft=False)
    ax.set_title('CEF', fontsize=7, pad=3)
    ax.set_xlabel('$-\\Delta\\theta$ (deg)')
    ax.set_xlim(0, np.arccos(1.0 / np.sqrt(3.0)) * 180 / np.pi - 50.86)
    ax.set_xticks(np.arange(0.0, np.arccos(1.0 / np.sqrt(3.0)) * 180 / np.pi - 50.86, 2.0))
    ax.set_ylim(-0.2, 2.25)
    ax.set_yticks(np.arange(0.0, 2.1, 0.5))

def plot_hopping(ax):
    # Preserve the published draw order: 20 untouched raw branches followed by
    # the 16 manually corrected replacements.
    for i in np.concatenate([fig5_raw_hopping, fig5_corrected]):
        ax.plot(fig5_x, fig5_y[int(i)], c=ctab[int(i)])
    ax.invert_xaxis()
    make_theme(ax, ticklength=4)
    ax.tick_params(labelleft=False)
    ax.set_title('Hopping', fontsize=7, pad=3)
    ax.set_xlabel('$-V_{dd}$ (eV)')
    ax.set_xlim(0.0, 1.33)
    ax.set_xticks(np.arange(0.0, 1.33, 0.3))
    ax.set_ylim(-0.2, 2.25)
    ax.set_yticks(np.arange(0.0, 2.1, 0.5))



def plot_RIXS(ax):
    rixs4 = Cmap_calc[3]['y']
    rixs3 = Cmap_calc[2]['y']
    eloss = Cmap_calc[2]['x']
    choose = np.logical_and(eloss >= -0.5, eloss <= 2.25)
    ax.plot(rixs4[choose], eloss[choose], '-', lw=1, color='C5', label=Cmap_calc[3]['Q'])
    ax.plot(rixs3[choose], eloss[choose], '-', lw=1, color='C6', label=Cmap_calc[2]['Q'])
    make_theme(ax, ticklength=4)
    ax.tick_params(labelleft=False)
    ax.set_title('RIXS simulation', fontsize=7, pad=3)
    ax.set_xlabel('Intensity (arb. units)')
    ax.set_xlim(-0.1, 0.8)
    ax.set_xticks(np.arange(0.0, 1.201, 0.3))
    ax.legend(loc='upper right', frameon=False, handletextpad=0.3, labelspacing=0.2, fontsize=6)
    ax.set_ylim(-0.2, 2.25)
    ax.set_yticks(np.arange(0.0, 2.1, 0.5))

def plot_level(ax):
    lw = 1
    dash_params = dict(ls='-', color='gray', zorder=-1, lw=0.5)
    ax.plot([-16.5, -13.5], [1.0, 1.0], '-', lw=lw, color='k')
    ax.plot([13.5, 16.5], [1.0, 1.0], '-', lw=lw, color='k')
    evals = eigvals_all0[29, :36] - eigvals_all0[29][0]
    evals = np.round(evals, 6)
    energy1A = np.mean(evals[:4]) / 2
    energy1B = np.mean(evals[4:20]) - energy1A
    GS = -1.0 + (energy1A * 2 + energy1B * 4) / 6
    energy1A = energy1A - GS
    energy1B = energy1B - GS
    ax.plot([-11.5, -8.5], [energy1A, energy1A], '-', lw=lw, color='C3')
    ax.plot([11.5, 8.5], [energy1A, energy1A], '-', lw=lw, color='C3')
    ax.plot([-13.5, -11.5], [1.0, energy1A], **dash_params)
    ax.plot([13.5, 11.5], [1.0, energy1A], **dash_params)
    ax.text(-10, energy1A + 0.06, '$J$=1/2', color='C3', horizontalalignment='center', fontsize=6)
    ax.text(10, energy1A + 0.06, '$J$=1/2', color='C3', horizontalalignment='center', fontsize=6)
    ax.plot([-11.5, -8.5], [energy1B, energy1B], '-', lw=lw, color='C0')
    ax.plot([11.5, 8.5], [energy1B, energy1B], '-', lw=lw, color='C0')
    ax.plot([-13.5, -11.5], [1.0, energy1B], **dash_params)
    ax.plot([13.5, 11.5], [1.0, energy1B], **dash_params)
    ax.text(-10, energy1B + 0.06, '$J$=3/2', color='C0', horizontalalignment='center', fontsize=6)
    ax.text(10, energy1B + 0.06, '$J$=3/2', color='C0', horizontalalignment='center', fontsize=6)
    evals = eigvals_all0[59, :36] - eigvals_all0[59][0]
    evals = np.round(evals, 6)
    energy2A = np.mean(evals[:4]) / 2
    energy2B = np.mean(evals[4:12]) - energy2A
    energy2C = np.mean(evals[12:20]) - energy2A
    GS = -1.0 + np.mean([energy2A, energy2B, energy2C])
    energy2A = energy2A - GS
    energy2B = energy2B - GS
    energy2C = energy2C - GS
    ax.plot([-6.5, -3.5], [energy2A, energy2A], '-', lw=lw, color='C3')
    ax.plot([6.5, 3.5], [energy2A, energy2A], '-', lw=lw, color='C3')
    ax.plot([-8.5, -6.5], [energy1A, energy2A], **dash_params)
    ax.plot([8.5, 6.5], [energy1A, energy2A], **dash_params)
    ax.plot([-6.5, -3.5], [energy2B, energy2B], '-', lw=lw, color='C0')
    ax.plot([6.5, 3.5], [energy2B, energy2B], '-', lw=lw, color='C0')
    ax.plot([-8.5, -6.5], [energy1B, energy2B], **dash_params)
    ax.plot([8.5, 6.5], [energy1B, energy2B], **dash_params)
    ax.plot([-6.5, -3.5], [energy2C, energy2C], '-', lw=lw, color='C0')
    ax.plot([6.5, 3.5], [energy2C, energy2C], '-', lw=lw, color='C0')
    ax.plot([-8.5, -6.5], [energy1B, energy2C], **dash_params)
    ax.plot([8.5, 6.5], [energy1B, energy2C], **dash_params)
    energy3A = 0
    ax.plot([-1.5, 1.5], [energy3A, energy3A], '-', lw=lw, color='C3')
    ax.plot([-3.5, -1.5], [energy2A, energy3A], **dash_params)
    ax.plot([3.5, 1.5], [energy2A, energy3A], **dash_params)
    ax.text(0, energy3A + 0.06, '$J_{eff}$=0', color='C3', horizontalalignment='center', fontsize=6)
    for i in fig5_raw_level:
        i = int(i)
        energy = fig5_end_energy[i]
        if typetab[i] == 0:
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C3')
            ax.plot([-3.5, -1.5], [energy2A, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2A, energy], **dash_params)
            if i == 0:
                ax.text(
                    0,
                    energy - 0.12,
                    '$J_{eff}$=1',
                    color='C3',
                    horizontalalignment='center',
                    fontsize=6,
                )
        if i in range(4, 12):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C4')
            ax.plot([-3.5, -1.5], [energy2A, energy], **dash_params)
            ax.plot([-3.5, -1.5], [energy2B, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2A, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2B, energy], **dash_params)
        if i in range(12, 20):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C4')
            ax.plot([-3.5, -1.5], [energy2A, energy], **dash_params)
            ax.plot([-3.5, -1.5], [energy2C, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2A, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2C, energy], **dash_params)
        if i in range(20, 24):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C0')
            ax.plot([-3.5, -1.5], [energy2B, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2B, energy], **dash_params)
        if i in range(24, 32):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C0')
            ax.plot([-3.5, -1.5], [energy2B, energy], **dash_params)
            ax.plot([-3.5, -1.5], [energy2C, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2B, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2C, energy], **dash_params)
        if i in range(32, 36):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C0')
            ax.plot([-3.5, -1.5], [energy2C, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2C, energy], **dash_params)
    for i in fig5_corrected:
        i = int(i)
        energy = fig5_end_energy[i]
        if i in range(4, 12):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C4')
            ax.plot([-3.5, -1.5], [energy2A, energy], **dash_params)
            ax.plot([-3.5, -1.5], [energy2B, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2A, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2B, energy], **dash_params)
        if i in range(12, 20):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C4')
            ax.plot([-3.5, -1.5], [energy2A, energy], **dash_params)
            ax.plot([-3.5, -1.5], [energy2C, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2A, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2C, energy], **dash_params)
        if i in range(20, 24):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C0')
            ax.plot([-3.5, -1.5], [energy2B, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2B, energy], **dash_params)
        if i in range(24, 32):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C0')
            ax.plot([-3.5, -1.5], [energy2B, energy], **dash_params)
            ax.plot([-3.5, -1.5], [energy2C, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2B, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2C, energy], **dash_params)
        if i in range(32, 36):
            ax.plot([-1.5, 1.5], [energy, energy], '-', lw=lw, color='C0')
            ax.plot([-3.5, -1.5], [energy2C, energy], **dash_params)
            ax.plot([3.5, 1.5], [energy2C, energy], **dash_params)
    make_theme(ax, minorx=False, ticklength=4)
    ax.tick_params(top=False)
    ax.plot(
        [-17.0, -3.0, -3.0, -17.0, -17.0],
        [0.4, 0.4, 1.5, 1.5, 0.4],
        lw=0.5,
        ls='--',
        color='k',
        zorder=100,
    )
    ax.plot(
        [17.0, 3.0, 3.0, 17.0, 17.0],
        [0.4, 0.4, 1.5, 1.5, 0.4],
        lw=0.5,
        ls='--',
        color='k',
        zorder=100,
    )
    ax.text(-10.0, -0.8, 'Ir1', color='k', horizontalalignment='center')
    ax.text(10.0, -0.8, 'Ir2', color='k', horizontalalignment='center')
    ax.set_xlim(-19, 19)
    ax.set_xticks([-12.5, -7.5, 0.0, 7.5, 12.5])
    ax.set_xticklabels(['SOC', 'CEF', 'Hopping', 'CEF', 'SOC'])
    ax.set_ylabel('Energy (eV)')
    ax.set_ylim(-0.2, +2.25)
    ax.set_yticks(np.arange(-0.0, 2.26, 0.5))


## Main Fig. 2 — Ce incident-energy-dependent RIXS map

Measurement and simulation are compared using the incident-energy data and model grids prepared above. The simulation uses the incident-energy normalization of Ce fit **0**. The fixed momentum is Q=(−0.5, 0, 18.94) r.l.u.; the vertical line marks 11.216 keV.

The measurement and simulation use different color maxima (`0.5` and `0.62`), so equal colors do not represent identical intensities across the two panels.

Originally plotted with Matplotlib 3.11.1


In [ ]:
paper_rc_default9()

# Figure-layout state verified against the original production notebook.
fig = plt.figure(figsize=(6.75 / 2, 6.75 / 2 / 4 * 3), dpi=150)
gs1 = fig.add_gridspec(nrows=1, ncols=1, left=0.1, right=0.49, top=0.92, bottom=0.12)
gs2 = fig.add_gridspec(nrows=1, ncols=1, left=0.51, right=0.87, top=0.92, bottom=0.12)
gs3 = fig.add_gridspec(nrows=1, ncols=1, left=0.92, right=0.94, top=0.7, bottom=0.3)
axs = [fig.add_subplot(gs) for gs in gs1] + [fig.add_subplot(gs) for gs in gs2]
axs += [fig.add_subplot(gs) for gs in gs3]

_, art = plot_image(
    Emap_data['intensity'],
    x=Emap_data['omega'],
    y=Emap_data['eloss'],
    ax=axs[0],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.5,
)
_, art = plot_image(
    test_Edep * scale_factor_E[0],
    x=(ce_incident_energy_grid + 11774.5) / 1000,
    y=ce_energy_loss_grid,
    ax=axs[1],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.62,
)
cbar = plt.colorbar(art, ax=axs[1], cax=axs[2])

lim = cbar.ax.get_ylim()
cbar.outline.set_linewidth(0.5)
cbar.ax.tick_params(which='major', width=0.5, labelright=False, length=0)
cbar.ax.set_yticks(lim)
cbar.set_label('Intensity (arb. units)', labelpad=20, rotation=-90)

fmt = '{:.2g}'
lo, hi = lim
cbar.ax.text(0.5, -0.07, fmt.format(lo), transform=cbar.ax.transAxes, ha='center', va='center')
cbar.ax.text(0.5, 1.0 + 0.07, fmt.format(hi), transform=cbar.ax.transAxes, ha='center', va='center')

axs[1].set_xlim(axs[0].get_xlim())

for ax in axs[:2]:
    make_theme(ax, ticklength=4)
    ax.set_ylim(-0.4, 2)
    ax.set_xlim(11.212, ax.get_xlim()[1])
    ax.set_yticks(np.arange(0.0, 2.1, 1.0))
    ax.axvline(x=11.216, linestyle='--', linewidth=1, color='black')

axs[1].tick_params(labelleft=False)
axs[0].set_ylabel('Energy loss (eV)')
plot_label(axs[0], 'a', x=0.05, y=1.04, color='k')
axs[0].text(
    0.5,
    1.04,
    'Measurement',
    color='k',
    transform=axs[0].transAxes,
    horizontalalignment='center',
    verticalalignment='center',
)
plot_label(axs[1], 'b', x=0.05, y=1.04, color='k')
axs[1].text(
    0.5,
    1.04,
    'Simulation',
    color='k',
    transform=axs[1].transAxes,
    horizontalalignment='center',
    verticalalignment='center',
)

p0 = axs[0].get_position()
p1 = axs[1].get_position()
xmid = 0.5 * (p0.x0 + p1.x1)
y = p0.y0 - 0.08

fig.text(xmid, y, 'Incident energy (keV)', ha='center', va='top')

for ax in axs[:2]:
    xmin, xmax = [11.214, 11.218]
    ax.set_xticks([xmin, xmax])
    ax.set_xticklabels([f'{xmin:.4f}', f'{xmax:.4f}'])

fig.savefig('Ce_EnergyMap_new.pdf', bbox_inches='tight', pad_inches=0.02)


## Main Fig. 3 — Ce momentum-dependent RIXS map

Measurement and simulation are compared as functions of momentum and energy loss. The simulation uses Ce fit **0** and its momentum-derived intensity scale. Both panels share the color range `0–0.7`. The vertical line marks L=18.94; the top and bottom axes express the same momentum in the two reciprocal-space conventions.

Originally plotted with Matplotlib 3.11.1


In [ ]:
paper_rc_default9()

# Figure-layout state verified against the original production notebook.
fig = plt.figure(figsize=(6.75 / 2, 6.75 / 2), dpi=150)
gs1 = fig.add_gridspec(nrows=1, ncols=1, left=0.12, right=0.89, top=0.94, bottom=0.55)
gs2 = fig.add_gridspec(nrows=1, ncols=1, left=0.12, right=0.89, top=0.49, bottom=0.1)
gs3 = fig.add_gridspec(nrows=1, ncols=1, left=0.92, right=0.94, top=0.72, bottom=0.32)
axs = [fig.add_subplot(gs) for gs in gs1] + [fig.add_subplot(gs) for gs in gs2]
axs += [fig.add_subplot(gs) for gs in gs3]

_, art = plot_image(
    Ldep_data['intensity'],
    x=Ldep_data['HKL'][:, 2] * Lfactor,
    y=Ldep_data['eloss'],
    ax=axs[0],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.7,
)
_, art = plot_image(
    test * scale_factor[0],
    x=Ls0 * Lfactor,
    y=cfg.eloss0,
    ax=axs[1],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.7,
)
cbar = plt.colorbar(art, ax=axs[1], cax=axs[2])
format_cbar(cbar, padding=0.07)

for ax in axs[:2]:
    make_theme(ax, ticklength=4)
    ax.axhline(y=0.0, ls='--', lw=0.5, color='k', zorder=100)
    ax.axvline(x=L * Lfactor, linestyle='--', linewidth=1, color='black')
    ax.set_ylim(-0.25, 2)
    ax.set_yticks(np.arange(0.0, 2.1, 0.5))
    ax.set_xlim(1.12, 3.36)
    ax.set_xticks(np.arange(1.5, 3.5, 0.5))
    ax.set_ylabel('Energy loss (eV)')

axs[0].tick_params(labelbottom=False)
axs[1].set_xlabel('$Q_L$ ($2\\pi/d$)')
plot_label(axs[0], 'a', x=0.04, y=1.06, color='k')
axs[0].set_title('Measurement', fontsize=7, pad=3)
plot_label(axs[1], 'b', x=0.04, y=1.06, color='k')
axs[1].set_title('Simulation', fontsize=7, pad=3)
axs_top = axs[0].twiny()
axs_top.set_xlim(1.12 / Lfactor, 3.36 / Lfactor)
axs_top.set_xlabel('$Q_L$ ($2\\pi/c_L$)')
axs_top.tick_params(direction='in')
axs_top.set_xticks(np.arange(8, 20, 4.0))
axs_top = axs[1].twiny()
axs_top.set_xlim(1.12 / Lfactor, 3.36 / Lfactor)
axs_top.set_xticks(np.arange(8, 20, 4.0))
axs_top.tick_params(labeltop=False, direction='in')

fig.savefig('Ce_LMap_new.pdf', bbox_inches='tight', pad_inches=0.02)


## Main Fig. 4 — Ce representative RIXS cuts

The six selected spectra are compared with their corresponding simulated cuts after background subtraction and self-absorption correction. Experimental error bars are the stored intensity uncertainties divided by the absorption factor. The legend displays the magnitude of the second HKL component; the underlying geometry values are unchanged.

Originally plotted with Matplotlib 3.9.4


In [ ]:
paper_rc_default9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    2,
    1,
    sharex=True,
    sharey=True,
    figsize=(6.75 / 2, 6.75 / 2),
    dpi=150,
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

for scan1, scan2, color in zip(Cmap_data, Cmap_calc, ['k', 'C1', 'C0', 'C3', 'C2', 'C4']):
    choose = scan1['x'] <= 2.6
    axs[0].errorbar(
        scan1['x'][choose],
        scan1['y'][choose],
        scan1['yerr'][choose],
        fmt='o',
        color=color,
        ls='--',
        lw=0.5,
        ms=1,
        label='$Q$=({:.1f}, {:.0f}, {:.2f})'.format(
            scan1['HKLc'][0],
            abs(scan1['HKLc'][1]),
            scan1['HKLc'][2],
        ),
    )
    choose = scan2['x'] <= 2.6
    axs[1].plot(
        scan2['x'][choose],
        scan2['y'][choose],
        '-',
        lw=1,
        color=color,
        label='$Q$=' + scan2['Q'],
    )

axs[0].legend(
    loc='best',
    frameon=False,
    handletextpad=0.3,
    labelspacing=0.2,
    fontsize=6,
    title='Measurement',
)
axs[1].legend(
    loc='best',
    frameon=False,
    handletextpad=0.3,
    labelspacing=0.2,
    fontsize=6,
    title='Simulation',
)

for ax in axs:
    make_theme(ax, ticklength=4)
    ax.axhline(y=0.0, ls='--', lw=0.5, color='k', zorder=100)

axs[0].set_xlim(-0.25, 2.5)
axs[0].set_xticks(np.arange(0.0, 2.01, 0.5))
axs[0].set_ylim(-0.1, 1.5)
axs[0].set_yticks(np.arange(0.0, 1.21, 0.4))
axs[1].set_xlabel('Energy loss (eV)')
axs[0].set_ylabel('Intensity (arb. units)')
axs[0].yaxis.set_label_coords(-0.09, 0.0)
plot_label(axs[0], 'a', x=0.04, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.04, y=0.9, color='k')
plt.savefig('Ce_CutMap_new.pdf')


## Main Fig. 5 — Ce level evolution

**Deterministic branch reconstruction.** The historical min-cost-flow object was
not serialized and is not stable under tied/degenerate assignments. The branch
trajectories and hopping endpoint energies used here are frozen from the
published vector figure in `Ce_Fig5_published_branches.npz`.


The five panels assemble the SOC, crystal-field, hopping, RIXS-cut, and schematic level calculations prepared above. The level diagram summarizes the same progression and identifies the singlet ground state and triplet excitation discussed in the main manuscript.

Originally plotted with Matplotlib 3.9.4


In [ ]:
paper_rc_level7()

# Figure-layout state verified against the original production notebook.
fig = plt.figure(figsize=(6.75, 6.75 / 3), dpi=150)
gs1 = fig.add_gridspec(nrows=1, ncols=1, left=0.06, right=0.14, top=0.93, bottom=0.14)
gs2 = fig.add_gridspec(nrows=1, ncols=1, left=0.14, right=0.22, top=0.93, bottom=0.14)
gs3 = fig.add_gridspec(nrows=1, ncols=1, left=0.22, right=0.38, top=0.93, bottom=0.14)
gs4 = fig.add_gridspec(nrows=1, ncols=1, left=0.38, right=0.54, top=0.93, bottom=0.14)
gs5 = fig.add_gridspec(nrows=1, ncols=1, left=0.63, right=0.98, top=0.93, bottom=0.14)
axs = [fig.add_subplot(gs) for gs in gs1] + [fig.add_subplot(gs) for gs in gs2]
axs += [fig.add_subplot(gs) for gs in gs3] + [fig.add_subplot(gs) for gs in gs4]
axs += [fig.add_subplot(gs) for gs in gs5]

plot_SOC(axs[0])
plot_label(axs[0], 'a', x=0.1, y=1.04, color='k')

plot_CEF(axs[1])
plot_label(axs[1], 'b', x=0.1, y=1.04, color='k')

plot_hopping(axs[2])
plot_label(axs[2], 'c', x=0.06, y=1.04, color='k')

plot_RIXS(axs[3])
plot_label(axs[3], 'd', x=0.06, y=1.04, color='k')

plot_level(axs[4])

axs[4].plot(np.nan, np.nan, '-', lw=1, color='C0', label='$|{3/2}\\rangle\\otimes |{3/2}\\rangle$')
axs[4].plot(np.nan, np.nan, '-', lw=1, color='C4', label='$|{1/2}\\rangle\\otimes |{3/2}\\rangle$')
axs[4].plot(np.nan, np.nan, '-', lw=1, color='C3', label='$|{1/2}\\rangle\\otimes |{1/2}\\rangle$')
axs[4].legend(loc='upper right', frameon=False, fontsize=6, handletextpad=0.3, labelspacing=0.2)
plot_label(axs[4], 'e', x=0.05, y=1.04, color='k')

plt.savefig('level_new.pdf', bbox_inches='tight', pad_inches=0.01)


## Main Fig. 6 and Supplementary Fig. S2 — Ce momentum-dependent QFI

**Main Fig. 6.** The left panel compares the representative theoretical nQFI from Ce fit **0** with the mean and standard deviation of the 12-member experimental/model ensemble. The right panel compares unpolarized and outgoing-polarization-resolved results. The red line at nQFI=1 is the entanglement threshold; the gray line identifies the fixed momentum of the incident-energy scan.

Fig. 6 plotted with Matplotlib 3.11.1; Fig. S2 plotted with Matplotlib 3.8.0


In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    1,
    2,
    dpi=150,
    sharey=True,
    figsize=(6.75 / 2, 6.75 * 2 / 9),
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

x = Ls[:][choose0] * Lfactor
y = QFI_mean
yerr = [QFI_std, QFI_std]
axs[0].plot(newmomenta[:], theorSpectra[0], label='theory')
axs[0].errorbar(x, y, yerr=yerr, fmt='*', capsize=3, label='meas.+model')
axs[0].axvline(x=L * Lfactor, linestyle='--', linewidth=1, color='gray')

axs[1].plot(newmomenta, theorSpectra[0], label=f'unpolarized')
axs[1].plot(newmomenta, theorSpectraM0[0], label=f'in $\\pi$, out $\\pi$')
axs[1].plot(newmomenta, theorSpectraMpi2[0], label=f'in $\\pi$, out $\\sigma$')
axs[1].axvline(x=L * Lfactor, linestyle='--', linewidth=1, color='gray')

axs[0].legend(fontsize=6, loc=(0.1, 0.77), frameon=False)
axs[1].legend(fontsize=6, loc=(0.13, 0.65), frameon=False)

for ax in axs:
    make_theme(ax, ticklength=4)

axs[0].set_ylim(0.0, 1.5)
axs[0].set_ylabel('nQFI')

for i in range(2):
    axs[i].set_xlim(-0.2, 3.7)
    axs[i].set_xticks(np.arange(0, 3.1, 1))
    axs[i].set_xlabel('$Q_L$ $(2\\pi/d)$')
    axs[i].axhline(y=1, color='r', linestyle='--', lw=0.5)
    axs_top = axs[i].twiny()
    axs_top.set_xlim(-0.2 / Lfactor, 3.7 / Lfactor)
    axs_top.set_xticks(np.arange(0, 18.1, 6))
    axs_top.tick_params(direction='in')
    axs_top.set_xlabel('$Q_L$ ($2\\pi/c_L$)')

plot_label(axs[0], 'a', x=0.07, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.07, y=0.9, color='k')

plt.savefig('Ce_exp_new.pdf', bbox_inches='tight', pad_inches=0.01)


**Supplementary Fig. S2.** The numerator and one-partite denominator underlying the nQFI curves are plotted separately for the unpolarized and polarization-resolved channels. Their quotient produces the corresponding normalized theoretical curves in Fig. 6.


In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    1,
    2,
    dpi=150,
    sharey=True,
    figsize=(6.75 / 2, 6.75 * 2 / 9),
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

axs[0].plot(newmomenta, theorSpectra[0] * denomTheor[0], label=f'unpolarized')
axs[0].plot(newmomenta, nominator1M, label=f'in $\\pi$, out $\\pi$')
axs[0].plot(newmomenta, nominator2M, label=f'in $\\pi$, out $\\sigma$')
axs[0].axvline(x=L * Lfactor, linestyle='--', linewidth=1, color='gray')

axs[1].plot(newmomenta, denomTheor[0], label=f'unpolarized')
axs[1].plot(newmomenta, nominator1M / theorSpectraM0[0], label=f'in $\\pi$, out $\\pi$')
axs[1].plot(newmomenta, nominator2M / theorSpectraMpi2[0], label=f'in $\\pi$, out $\\sigma$')
axs[1].axvline(x=L * Lfactor, linestyle='--', linewidth=1, color='gray')

axs[0].legend(fontsize=6, loc=(0.13, 0.65), frameon=False)
axs[1].legend(fontsize=6, loc='best', frameon=False)

for ax in axs:
    make_theme(ax, ticklength=4)

axs[0].set_ylim(0.0, 0.5)
axs[0].set_ylabel('QFI numerator')
axs[1].set_ylabel('QFI denominator')

for i in range(2):
    axs[i].set_xlim(-0.2, 3.7)
    axs[i].set_xticks(np.arange(0, 3.1, 1))
    axs[i].set_xlabel('$Q_L$ $(2\\pi/d)$')
    axs[i].axhline(y=1, color='r', linestyle='--', lw=0.5)
    axs_top = axs[i].twiny()
    axs_top.set_xlim(-0.2 / Lfactor, 3.7 / Lfactor)
    axs_top.set_xticks(np.arange(0, 18.1, 6))
    axs_top.tick_params(direction='in')
    axs_top.set_xlabel('$Q_L$ ($2\\pi/c_L$)')

plot_label(axs[0], 'a', x=0.07, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.07, y=0.9, color='k')

plt.savefig('Ce_exp_new_nonorm.pdf', bbox_inches='tight', pad_inches=0.01)


## Main Fig. 7 and Supplementary Fig. S3 — Ce incident-energy-dependent QFI

**Main Fig. 7.** The same comparison is made while varying incident energy at fixed momentum. The experimental/model points use the Ce ensemble prepared above, and the theoretical curve uses fit **0**. The vertical line marks the incident energy used for the momentum-dependent measurements.

Fig. 7 plotted with Matplotlib 3.11.1. Fig. S3 plotted with Matplotlib 3.9.4

In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    1,
    2,
    dpi=150,
    sharey=True,
    figsize=(6.75 / 2, 6.75 * 2 / 9),
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

x = omega_data
y = QFI_mean_E
yerr = [QFI_std_E, QFI_std_E]
axs[0].plot(omega11, theorSpectraE[0], label='theory')
axs[0].errorbar(x, y, yerr=yerr, fmt='*', capsize=3, label='meas.+model')
axs[0].axvline(x=11.216, linestyle='--', linewidth=1, color='gray')

axs[1].plot(omega11, theorSpectraE[0], label='unpolarized')
axs[1].plot(omega11, theorSpectraE0[0], label='in $\\pi$, out $\\pi$')
axs[1].plot(omega11, theorSpectraEpi2[0], label='in $\\pi$, out $\\sigma$')
axs[1].axvline(x=11.216, linestyle='--', linewidth=1, color='gray')

axs[0].legend(loc='best', fontsize=6, frameon=False)
axs[1].legend(loc='best', fontsize=6, frameon=False)

for ax in axs:
    make_theme(ax, ticklength=4)

axs[0].set_ylim(0, 1.5)
axs[0].set_yticks(np.arange(0, 1.6, 0.3))
axs[1].set_ylim(0, 1.5)
axs[0].set_xlabel('Incident energy (keV)')
axs[1].set_xlabel('Incident energy (keV)')
axs[0].set_ylabel('nQFI')
axs[0].axhline(y=1, color='r', linestyle='--', lw=0.5)
axs[1].axhline(y=1, color='r', linestyle='--', lw=0.5)
plot_label(axs[0], 'a', x=0.07, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.07, y=0.9, color='k')
plt.savefig('Ce_simulation_new.pdf', bbox_inches='tight', pad_inches=0.01)


**Supplementary Fig. S3.** The QFI numerator and one-partite denominator underlying the incident-energy-dependent normalized curves are shown separately.


In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    1,
    2,
    dpi=150,
    sharey=True,
    figsize=(6.75 / 2, 6.75 * 2 / 9),
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

axs[0].plot(omega11, theorSpectraE[0] * denomTheorE[0], label='unpolarized')
axs[0].plot(omega11, nominator1E, label='in $\\pi$, out $\\pi$')
axs[0].plot(omega11, nominator2E, label='in $\\pi$, out $\\sigma$')
axs[0].axvline(x=11.216, linestyle='--', linewidth=1, color='gray')

axs[1].plot(omega11, denomTheorE[0], label='unpolarized')
axs[1].plot(omega11, nominator1E / theorSpectraE0[0], label='in $\\pi$, out $\\pi$')
axs[1].plot(omega11, nominator2E / theorSpectraEpi2[0], label='in $\\pi$, out $\\sigma$')
axs[1].axvline(x=11.216, linestyle='--', linewidth=1, color='gray')

axs[0].legend(loc='best', fontsize=6, frameon=False)
axs[1].legend(loc='best', fontsize=6, frameon=False)

for ax in axs:
    make_theme(ax, ticklength=4)

axs[0].set_ylim(0, 0.6)
axs[0].set_yticks(np.arange(0, 0.6, 0.1))
axs[1].set_ylim(0, 0.6)
axs[0].set_xlabel('Incident energy (keV)')
axs[1].set_xlabel('Incident energy (keV)')
axs[0].set_ylabel('QFI numerator')
axs[1].set_ylabel('QFI denominator')
plot_label(axs[0], 'a', x=0.07, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.07, y=0.9, color='k')
plt.savefig('Ce_simulation_new_nonorm.pdf', bbox_inches='tight', pad_inches=0.01)


## Supplementary Fig. S4 — one-partite vs even/odd-parity maxima

**S4 state reconstruction from saved optima.** The optimization results are loaded from `Ce_t2g_maxwal_P.npz` and `Ce_t2g_maxwal_M.npz`; the maximization searches are performed by the script `compute_ce_parity_maxima.py`. Their physical construction is described in Supplementary Information, Secs. III C and IV C.

The fixed-total-occupancy Ce basis has ten valence electrons. The local t₂g occupancy sectors N=4, 5, and 6 contain 15, 6, and 1 states, respectively, giving the local dimension **22** used in `parity_matrix`. The explicit orbital permutations align the Fock and product-state bases for the site-exchange operation and its fermionic sign.

Diagonalizing the parity operator supplies the even-parity subspace (`w > 1−10⁻⁸`). The saved amplitude/phase parameters are converted to normalized states in this subspace, and `p55list` records their total probability in the (5,5) occupancy sector. The `10⁻¹²` threshold selects a nonzero component for fixing the irrelevant global phase.

Originally plotted with Matplotlib 3.8.0

In [ ]:
all_basis = [get_fock_bin_by_N_S(8, 0, 12, 10, dispinfo=False)]
basis = SparseBinary(np.vstack(all_basis), shape=[None, 20])
coo_table = np.hstack([0, 1, 6, 7, 10, 11, 16, 17, 2, 3, 4, 5, 8, 9, 12, 13, 14, 15, 18, 19])
basis.change_coordinates(coo_table, shape=[None, 20])
npbasis = basis.toarray()
weights = 1 << np.arange(19, -1, -1)
npbasis_ind = npbasis @ weights
local456 = np.concatenate([
    np.array(edrixs.get_fock_bin_by_N(6, 6, 4, 0), dtype=np.int64),
    np.array(edrixs.get_fock_bin_by_N(6, 5, 4, 0), dtype=np.int64),
    np.array(edrixs.get_fock_bin_by_N(6, 4, 4, 0), dtype=np.int64),
])[:, [6, 7, 0, 1, 2, 3, 8, 9, 4, 5]]
tp456 = np.array([np.concatenate([a, b]) for a in local456 for b in local456])
tp456_ind = tp456 @ weights
indlist456 = np.array([np.argwhere(npbasis_ind[i] == tp456_ind)[0] for i in range(len(npbasis_ind))]).ravel()

def parity_matrix(indlist, dloc=22):
    ind = np.asarray(indlist, int).ravel()
    pos = {int(v): i for i, v in enumerate(ind)}
    P = np.zeros((len(ind), len(ind)))
    for i, full in enumerate(ind):
        a, b = divmod(int(full), dloc)
        P[i, pos[dloc * b + a]] = 1.0
    return P
P = parity_matrix(indlist456) * (-1) ** np.sum(npbasis[:, :10], axis=1)
w, V = np.linalg.eigh((P + P.T.conj()) / 2)
Qp = V[:, w > 1 - 1e-08]

Pdat = np.load('Ce_t2g_maxwal_P.npz')
FQ_list3 = Pdat['FQ_list3']
arg_list = Pdat['arg_list']
FQ_listM = np.load('Ce_t2g_maxwal_M.npz')['FQ_listM']

def params_to_state(params):
    params = np.asarray(params)
    n = len(params) // 2
    u = np.asarray(params[:n], float)
    th = np.asarray(params[n:], float)
    r = u / np.linalg.norm(u)
    z = r * np.exp(1j * th)
    nz = np.flatnonzero(np.abs(z) > 1e-12)
    if len(nz):
        z *= np.exp(-1j * np.angle(z[nz[0]]))
    return Qp @ z
local5 = np.array(
    edrixs.get_fock_bin_by_N(6, 5, 4, 0),
    dtype=np.int64,
)[:, [6, 7, 0, 1, 2, 3, 8, 9, 4, 5]]
tp55 = np.array([np.concatenate([a, b]) for a in local5 for b in local5])
tp55_ind = tp55 @ weights
ind55 = np.array([np.argwhere(npbasis_ind == x)[0] for x in tp55_ind]).ravel()
veclist = np.array([params_to_state(p) for p in arg_list])
p55list = np.array([np.sum(np.abs(v[ind55]) ** 2) for v in veclist])


The upper panel compares saved even- and odd-parity QFI maxima with the one-partite maximum for the representative Ce fit. The gray curve gives the (5,5) weight of the reconstructed even-parity maximizing state. The lower panel divides each parity-sector maximum by the one-partite maximum (Supplementary Information, Fig. S4).


In [ ]:
paper_rc_s4()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    2,
    1,
    sharex=True,
    figsize=(6.75 / 1.5, 6.75 / 1.5),
    dpi=150,
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.15),
)
axs[0].plot(Ls * Lfactor, np.array(FQ_list3), label='even parity')
axs[0].plot(Ls * Lfactor, np.array(FQ_listM), label='odd parity')
axs[0].plot(Ls * Lfactor, denomTheor[0], label='one-partite')
axs[0].plot(Ls * Lfactor, p55list, color='gray', label='weight of 5-5 at max')
axs[0].legend(loc='best')
axs[0].set_ylim([0, 2.0])
axs[1].plot(
    Ls * Lfactor,
    np.array(FQ_list3) / denomTheor[0],
    label='Ratio: even parity/one-partite',
)
axs[1].plot(Ls * Lfactor, np.array(FQ_listM) / denomTheor[0], label='Ratio: odd parity/one-partite')
axs[1].axhline(y=1.0, linestyle='--', color='r')
axs[1].set_xlabel('$Q_L$ ($2\\pi/d$)')
axs[1].legend(loc='best')
axs[1].set_ylim([0, 3])
plot_label(axs[0], 'a', x=0.04, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.04, y=0.9, color='k')
plt.savefig('Ce_QFI_evenvsodd_ratio.pdf', bbox_inches='tight')


## Ta — shared preparation for Supplementary Figs. S5–S9

The Ta configuration and parameter table replace the Ce working variables. Ta has 11 valence electrons per dimer, and its model includes excitations involving the e_g manifold (Supplementary Information, Secs. V–VI).

Row **9** has the smallest overall fitting distance and is used for the representative model curves and maps. The polarization-resolved files contain only this selected fit: their row **0** corresponds to row **9** in the full Ta files.

All **24** fitted Ta parameter sets are retained because they did not require the physical filtering applied to Ce. Separate intensity scales are obtained from the momentum and incident-energy references.

The momentum conversion uses `d=2.7` Å and reciprocal-lattice factor `0.436029515` Å⁻¹. The thermal factor uses 11 K, and the incident-energy grid follows the stored offset convention used by the Ta QFI files.


In [ ]:
cfg = import_config('config_Ta_Eg.py')

coordsfin = np.loadtxt('Ba3TaIr2O9_collectedV.txt')
labels0 = list(cfg.greedy_bounds.keys())
cmap = []
cmapEL = []

for ind in range(len(coordsfin)):
    dp = {labels0[i]: coordsfin[ind, i] for i in range(len(labels0))}
    dp2 = {**dp, **cfg.fixed_params_greedy}
    cmap.append(cfg.rixs_funV(dp2))
    cmapEL.append(cfg.rixs_funV_EL(dp2))

scale_factor = np.array(
    [
        np.sum(np.abs(cfg.rixs_ref)) / np.sum(np.abs(cmap[q]))
        for q in range(len(coordsfin))
    ]
)
scale_factor_E = np.array(
    [
        np.sum(np.abs(cfg.rixs_ref_Emap_Large)) / np.sum(np.abs(cmapEL[q]))
        for q in range(len(coordsfin))
    ]
)
d = 2.7
Lfactor = 0.436029515 * d / (2 * np.pi)

mom = np.load('Ta_Eg_momdep_outputsV.npz')
theorSpectra = np.vstack([mom['theorSpectra']])
theorSpectraL2 = np.vstack([mom['theorSpectraL2']])
numNegTheor = np.vstack([mom['numNegTheor']])
numNegTheorL2 = np.vstack([mom['numNegTheorL2']])
denomTheor = np.vstack([mom['denomTheor']])
numeratorLarge = theorSpectraL2 * denomTheor - numNegTheorL2
Ls = np.arange(0.0, 19.4, 0.1)
newmomenta = Ls * Lfactor
T = 11.0
beta = 1 / (kB * T)

mom_pol = np.load('Ta_Eg_momdep_pol_outputsV.npz')
theorSpectraM0 = mom_pol['theorSpectraM0']
theorSpectraMpi2 = mom_pol['theorSpectraMpi2']
omega = np.linspace(-10, 10, 201) - 559
omega11 = (omega + 11774.5) / 1000

edep = np.load('Ta_Eg_Edep_outputsV.npz')
theorSpectra0 = edep['theorSpectra']
numNegTheor0 = edep['numNegTheor']
denomTheor0 = edep['denomTheor']

edep_pol = np.load('Ta_Eg_Edep_pol_outputsV.npz')
theorSpectraE0 = edep_pol['theorSpectraE0']
theorSpectraEpi2 = edep_pol['theorSpectraEpi2']


### Experimental QFI preparation (Figs. S8–S9)


**Momentum-dependent Ta nQFI (Fig. S8).** The explicit scan order is paired with `ta_momentum_indices = [95, 105, …, 193]`, which selects their calculated momentum-grid positions. This correspondence is specified directly rather than derived from scan metadata.

Corrected measured intensities are integrated over `0.3 < energy loss < 6.0` eV. The fitting data for this momentum series extend only to 2 eV according to Supplementary Information, Sec. V; the wider mask does not create data beyond the available scan range. The additional model term `numeratorLarge` supplies a contribution derived from the difference between precomputed numerator products.

All 24 fits are combined with both normalization estimates, giving **48** equally weighted estimates. Their mean and standard deviation provide the plotted points and error bars. The use of both normalizations is described in Supplementary Information, Sec. V.


In [ ]:
Mmap_data = dict(eloss=[], intensity=[])

for scan_no in [31, 30, 29, 28, 27, 26, 32, 25, 33, 24, 34, 23, 35, 22, 36, 21]:
    scan = dict(np.load(f'data_Ba3TaIr2O9/scan_{scan_no}.npz'))
    e = (scan['x'] - scan['xoffset']) / 1000
    ch = (e > 0.3) & (e < 6.0)
    Mmap_data['eloss'].append(e[ch])
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Mmap_data['intensity'].append(((scan['y'] - bg) / scan['ab_factor'])[ch])

ta_momentum_indices = np.array([95, 105, 115, 125, 135, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190, 193])
QFI_data_L2 = []
QFI_data_L2E = []

for q in range(len(coordsfin)):
    pos = []
    for i in range(len(Mmap_data['intensity'])):
        e = np.asarray(Mmap_data['eloss'][i])
        y = np.asarray(Mmap_data['intensity'][i])
        fac = 2 * np.tanh(beta * e / 2) * (1 - np.exp(-beta * e))
        pos.append(np.sum(y * fac * (e[1] - e[0])))
    pos = np.asarray(pos)
    neg = numNegTheor[q][ta_momentum_indices]
    pl = numeratorLarge[q][ta_momentum_indices]
    nm = pos + (neg + pl) * scale_factor[q]
    ne = pos + (neg + pl) * scale_factor_E[q]
    QFI_data_L2.append(nm / (denomTheor[q][ta_momentum_indices] * scale_factor[q]))
    QFI_data_L2E.append(ne / (denomTheor[q][ta_momentum_indices] * scale_factor_E[q]))

QFI_data = np.vstack([QFI_data_L2, QFI_data_L2E])
QFI_mean = np.mean(QFI_data, axis=0)
QFI_std = np.std(QFI_data, axis=0)


**Incident-energy-dependent Ta nQFI (Fig. S9).** Scans `[3, 6, 9, 12, 15, 18]` are corrected and integrated over `0.3 < energy loss < 6.0` eV, covering the higher-energy excitations included in the Ta model. The measured contribution is combined with the precomputed negative-energy contribution for each fit.

Only the incident-energy normalization is used: this data set covers the e_g manifold, while the momentum-dependent reference is restricted to lower energy losses (Supplementary Information, Sec. V). The mean and standard deviation are therefore taken over **24** estimates. Incident energies are matched to the precomputed grid by exact lookup, as in the Ce calculation.


In [ ]:
Emap_data = dict(eloss=[], omega=[], intensity=[])

for scan_no in [3, 6, 9, 12, 15, 18]:
    scan = dict(np.load(f'data_Ba3TaIr2O9/scan_{scan_no}.npz'))
    e = (scan['x'] - scan['xoffset']) / 1000
    ch = (e > 0.3) & (e < 6.0)
    Emap_data['eloss'].append(e[ch])
    Emap_data['omega'].append(scan['Ei'])
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Emap_data['intensity'].append(((scan['y'] - bg) / scan['ab_factor'])[ch])

omega_data = np.array([x.item() if np.ndim(x) == 0 else x[0] for x in Emap_data['omega']])
omega_select = omega_data * 1000 - 11774.5
QFI_data_EdepE = []

for q in range(len(coordsfin)):
    pos = []
    for i in range(len(Emap_data['intensity'])):
        e = np.asarray(Emap_data['eloss'][i])
        y = np.asarray(Emap_data['intensity'][i])
        fac = 2 * np.tanh(beta * e / 2) * (1 - np.exp(-beta * e))
        pos.append(np.sum(y * fac * (e[1] - e[0])))
    pos_in_B = {val: idx for idx, val in enumerate(omega)}
    choose2 = np.array([pos_in_B[a] for a in omega_select])
    neg = numNegTheor0[q][choose2]
    ne = np.asarray(pos) + neg * scale_factor_E[q]
    QFI_data_EdepE.append(ne / (denomTheor0[q][choose2] * scale_factor_E[q]))

QFI_dataE = np.vstack(QFI_data_EdepE)
QFI_meanE = np.mean(QFI_dataE, axis=0)
QFI_stdE = np.std(QFI_dataE, axis=0)


### RIXS maps and cuts: model preparation (Figs. S5–S7)


**Ta momentum-map preparation (Fig. S6).** Experimental spectra are placed on a common energy grid. The dense simulation geometry is interpolated from the configuration's motor table, and the model is evaluated with Ta fit **9**. The motor-table columns supply HKL, scattering angle, and sample orientation; their definitions are part of the configuration.


In [ ]:
Ldep_data = dict(eloss=None, HKL=[], intensity=[])

for scan_no in [31, 30, 29, 28, 27, 26, 32, 25, 33, 24, 34, 23, 35, 22, 36, 21]:
    scan = dict(np.load(f'data_Ba3TaIr2O9/scan_{scan_no}.npz'))
    if Ldep_data['eloss'] is None:
        Ldep_data['eloss'] = scan['x'].copy()
    Ldep_data['HKL'].append(scan['HKL'])
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Ldep_data['intensity'].append(np.interp(
        Ldep_data['eloss'],
        scan['x'] - scan['xoffset'],
        (scan['y'] - bg) / scan['ab_factor'],
    ))

Ldep_data['intensity'] = np.vstack(Ldep_data['intensity']).T
Ldep_data['HKL'] = np.vstack(Ldep_data['HKL'])
Ldep_data['eloss'] *= 0.001
Ls0 = np.arange(6.0, 20.1, 0.1)
Hs0 = np.interp(Ls0, cfg.motors[:, 2], cfg.motors[:, 0])
Ks0 = np.interp(Ls0, cfg.motors[:, 2], cfg.motors[:, 1])
HKLs0 = np.vstack([Hs0, Ks0, Ls0]).T
tths0 = np.interp(Ls0, cfg.motors[:, 2], cfg.motors[:, 3])
ths0 = np.interp(Ls0, cfg.motors[:, 2], cfg.motors[:, 4])
chis0 = np.interp(Ls0, cfg.motors[:, 2], cfg.motors[:, 5])
phis0 = np.interp(Ls0, cfg.motors[:, 2], cfg.motors[:, 6])
Qs_vec0 = (cfg.Bmat @ HKLs0.T).T
Qs_mag0 = np.sqrt(np.sum(Qs_vec0 ** 2, axis=1))
cfg.geometry.update(dict(
    Ls=Ls0,
    HKLs=HKLs0,
    tths=tths0,
    ths=ths0,
    chis=chis0,
    phis=phis0,
    Qs_vec=Qs_vec0,
    Qs_mag=Qs_mag0,
))
cfg.resolution['eloss'] = Ldep_data['eloss']
dpbest = {labels0[i]: coordsfin[9, i] for i in range(len(labels0))}
dp2best = {**dpbest, **cfg.fixed_params_greedy}
pars = dict(
    Gam_c=dpbest['Gam_c'],
    sigma0=cfg.true_values['sigma0'],
    sigmas=cfg.sigmas,
    fraction=cfg.fraction,
    geometry=cfg.geometry,
    resolution=cfg.resolution,
    T=cfg.temperature,
    eline=None,
)
runner = RIXS_runner_Ta(
    pars,
    [cfg.sptab2i, cfg.sptab2n1, cfg.sptab2n2, cfg.sptab4i, cfg.sptab4n1, cfg.sptab4n2, cfg.sptabtr],
)
test = runner(dp2best)
xtest = Ldep_data['HKL'][:, 2] * Lfactor


**Ta representative-cut preparation (Fig. S7).** Scans `[49, 48, 41, 45]` are retained both as raw intensities and as background-subtracted, self-absorption-corrected spectra. Their individual geometries are used to calculate model cuts with fit **9**, scaled by its incident-energy normalization.

The explicit HKL labels in `Cmap_calc` must match the scan order. The calculation obtains the model geometry from the scan metadata and interpolated motor settings.


In [ ]:
Cmap_data = []
Cmap_data_raw = []

for scan_no in [49, 48, 41, 45]:
    scan = dict(np.load(f'data_Ba3TaIr2O9/scan_{scan_no}.npz'))
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    info = [scan['HKL'][0], scan['HKL'][1], scan['HKL'][2], scan['qc']]
    Cmap_data.append(dict(
        x=(scan['x'] - scan['xoffset']) / 1000.0,
        y=(scan['y'] - bg) / scan['ab_factor'],
        yerr=scan['yerr'] / scan['ab_factor'],
        HKLc=info,
    ))
    Cmap_data_raw.append(dict(
        x=(scan['x'] - scan['xoffset']) / 1000.0,
        y=scan['y'],
        yerr=scan['yerr'],
        HKLc=info,
    ))

scale_factorT = scale_factor_E[9]
cfg.resolution['eloss_Emap_large'] = Ldep_data['eloss']
test1 = []

for dat in Cmap_data:
    Lscan = dat['HKLc'][2]
    cfg.geometry['omega_Emap'] = np.array([-559.0])
    cfg.geometry['HKL_Emap'] = dat['HKLc'][:3]
    cfg.geometry['tth_Emap'] = float(np.interp(Lscan, cfg.motors_E[:, 2], cfg.motors_E[:, 3]))
    cfg.geometry['th_Emap'] = float(np.interp(Lscan, cfg.motors_E[:, 2], cfg.motors_E[:, 4]))
    cfg.geometry['chi_Emap'] = float(np.interp(Lscan, cfg.motors_E[:, 2], cfg.motors_E[:, 5]))
    cfg.geometry['phi_Emap'] = float(np.interp(Lscan, cfg.motors_E[:, 2], cfg.motors_E[:, 6]))
    pars = dict(
        Gam_c=dp2best['Gam_c'],
        sigma0=cfg.true_values['sigma0'],
        sigmas=cfg.sigmas,
        fraction=cfg.fraction,
        geometry=cfg.geometry,
        resolution=cfg.resolution,
        T=cfg.temperature,
        eline=None,
    )
    runner = RIXS_runner_Ta(
        pars,
        [
            cfg.sptab2i,
            cfg.sptab2n1,
            cfg.sptab2n2,
            cfg.sptab4i,
            cfg.sptab4n1,
            cfg.sptab4n2,
            cfg.sptabtr,
        ],
        mode='EL',
    )
    test1.append(runner(dp2best))

test2 = np.hstack(test1)
Cmap_calc = [
    dict(
        x=cfg.resolution['eloss_Emap_large'],
        y=test2[:, 0] * scale_factorT,
        HKLc=[-1.7, -1.1, 16.21],
    ),
    dict(
        x=cfg.resolution['eloss_Emap_large'],
        y=test2[:, 1] * scale_factorT,
        HKLc=[-1.72, -1.08, 13.52],
    ),
    dict(
        x=cfg.resolution['eloss_Emap_large'],
        y=test2[:, 2] * scale_factorT,
        HKLc=[-0.21, -0.31, 16.15],
    ),
    dict(
        x=cfg.resolution['eloss_Emap_large'],
        y=test2[:, 3] * scale_factorT,
        HKLc=[-0.19, -0.33, 18.84],
    ),
]


**Ta incident-energy-map preparation (Fig. S5).** The configured incident-energy measurement geometry is restored, the six measured spectra are interpolated onto a common energy-loss grid, and the model is evaluated over 50 incident energies. The energy-loss grid extends to approximately 6 eV to include the modeled higher-energy excitations.


In [ ]:
cfg.resolution['eloss_Emap_large'] = np.arange(-0.4, 5.96, 0.01)
cfg.geometry['HKL_Emap'] = cfg.HKL_Emap
cfg.geometry['tth_Emap'] = cfg.tth_Emap
cfg.geometry['th_Emap'] = cfg.th_Emap
cfg.geometry['chi_Emap'] = cfg.chi_Emap
cfg.geometry['phi_Emap'] = cfg.phi_Emap
Emap_data = dict(eloss=[], omega=[], intensity=[])

for scan_no in [3, 6, 9, 12, 15, 18]:
    scan = dict(np.load(f'data_Ba3TaIr2O9/scan_{scan_no}.npz'))
    e = (scan['x'] - scan['xoffset']) / 1000
    Emap_data['eloss'].append(e)
    Emap_data['omega'].append(scan['Ei'])
    bg = np.mean(scan['y'][(scan['x'] >= 250.0) & (scan['x'] <= 350.0)])
    Emap_data['intensity'].append(np.interp(
        cfg.resolution['eloss_Emap_large'],
        e,
        (scan['y'] - bg) / scan['ab_factor'],
    ))

Emap_data['intensity'] = np.vstack(Emap_data['intensity']).T
omega_data = np.array([x.item() if np.ndim(x) == 0 else x[0] for x in Emap_data['omega']])
cfg.geometry['omega_Emap'] = np.linspace(-563.5, -554.5, 50)
pars = dict(
    Gam_c=dp2best['Gam_c'],
    sigma0=cfg.true_values['sigma0'],
    sigmas=cfg.sigmas,
    fraction=cfg.fraction,
    geometry=cfg.geometry,
    resolution=cfg.resolution,
    T=cfg.temperature,
    eline=None,
)
runner_E = RIXS_runner_Ta(
    pars,
    [cfg.sptab2i, cfg.sptab2n1, cfg.sptab2n2, cfg.sptab4i, cfg.sptab4n1, cfg.sptab4n2, cfg.sptabtr],
    mode='EL',
)
test_Edep = runner_E(dp2best)


## Supplementary Fig. S5 — Ta incident-energy-dependent RIXS map

Measurement and simulation use the incident-energy normalization of Ta fit **9** and a common color range `0–0.7`. The vertical line marks the fixed incident energy, 11.2155 keV, used for the momentum-dependent series.

Originally plotted with Matplotlib 3.8.0


In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig = plt.figure(figsize=(6.75 / 2, 6.75 / 2 / 4 * 3), dpi=150)
gs1 = fig.add_gridspec(nrows=1, ncols=1, left=0.1, right=0.49, top=0.92, bottom=0.12)
gs2 = fig.add_gridspec(nrows=1, ncols=1, left=0.51, right=0.9, top=0.92, bottom=0.12)
gs3 = fig.add_gridspec(nrows=1, ncols=1, left=0.93, right=0.95, top=0.7, bottom=0.3)
axs = [fig.add_subplot(gs) for gs in gs1] + [fig.add_subplot(gs) for gs in gs2]
axs += [fig.add_subplot(gs) for gs in gs3]

_, art = plot_image(
    Emap_data['intensity'],
    x=omega_data,
    y=cfg.resolution['eloss_Emap_large'],
    ax=axs[0],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.7,
)
_, art = plot_image(
    test_Edep * scale_factor_E[9],
    x=(cfg.geometry['omega_Emap'] + 11774.5) / 1000,
    y=cfg.resolution['eloss_Emap_large'],
    ax=axs[1],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.7,
)
cbar = plt.colorbar(art, ax=axs[1], cax=axs[2])
format_cbar(cbar, padding=0.1)
cbar.set_label('Intensity (arb. units)', labelpad=15)
axs[1].set_xlim(axs[0].get_xlim())

for ax in axs[:2]:
    make_theme(ax, ticklength=4)
    ax.set_ylim(-0.4, 6)
    ax.set_yticks(np.arange(0.0, 6.1, 1.0))
    ax.set_xticks(np.arange(11.212, 11.2191, 0.004))
    ax.axvline(x=11.2155, linestyle='--', linewidth=1, color='black')

axs[1].tick_params(labelleft=False)
axs[0].set_xlabel('Incident energy (keV)')
axs[0].xaxis.set_label_coords(1.028, -0.09)
axs[0].set_ylabel('Energy loss (eV)')
plot_label(axs[0], 'a', x=0.05, y=1.04, color='k')
axs[0].text(
    0.5,
    1.04,
    'Measurement',
    color='k',
    transform=axs[0].transAxes,
    horizontalalignment='center',
    verticalalignment='center',
)
plot_label(axs[1], 'b', x=0.05, y=1.04, color='k')
axs[1].text(
    0.5,
    1.04,
    'Simulation',
    color='k',
    transform=axs[1].transAxes,
    horizontalalignment='center',
    verticalalignment='center',
)
fig.savefig('Ta_EnergyMap_new.pdf', bbox_inches='tight', pad_inches=0.02)


## Supplementary Fig. S6 — Ta momentum-dependent RIXS map

Measurement and simulation use Ta fit **9**, its momentum-derived normalization, and a common color range `0–0.7`. The vertical line marks L=18.4, the reference momentum used in the QFI comparisons.

Originally plotted with Matplotlib 3.8.0


In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig = plt.figure(figsize=(6.75 / 2, 6.75 / 2), dpi=150)
gs1 = fig.add_gridspec(nrows=1, ncols=1, left=0.12, right=0.89, top=0.94, bottom=0.55)
gs2 = fig.add_gridspec(nrows=1, ncols=1, left=0.12, right=0.89, top=0.49, bottom=0.1)
gs3 = fig.add_gridspec(nrows=1, ncols=1, left=0.92, right=0.94, top=0.72, bottom=0.32)
axs = [fig.add_subplot(gs) for gs in gs1] + [fig.add_subplot(gs) for gs in gs2]
axs += [fig.add_subplot(gs) for gs in gs3]

_, art = plot_image(
    Ldep_data['intensity'],
    x=Ldep_data['HKL'][:, 2] * Lfactor,
    y=Ldep_data['eloss'],
    ax=axs[0],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.7,
)
_, art = plot_image(
    test * scale_factor[9],
    x=Ls0 * Lfactor,
    y=Ldep_data['eloss'],
    ax=axs[1],
    cmap=turbo_w(),
    vmin=0.0,
    vmax=0.7,
)
cbar = plt.colorbar(art, ax=axs[1], cax=axs[2])
format_cbar(cbar, padding=0.07)

for ax in axs[:2]:
    make_theme(ax, ticklength=4)
    ax.axhline(y=0.0, ls='--', lw=0.5, color='k', zorder=100)
    ax.set_ylim(-0.25, 2)
    ax.set_yticks(np.arange(0.0, 2.1, 0.5))
    ax.set_xlim(9.6 * Lfactor, 19 * Lfactor)
    ax.set_xticks(np.arange(1.5, 4.2, 0.5))
    ax.set_ylabel('Energy loss (eV)')
    ax.axvline(x=18.4 * Lfactor, linestyle='--', linewidth=1, color='black')

axs[0].tick_params(labelbottom=False)
axs[1].set_xlabel('$Q_L$ ($2\\pi/d$)')
plot_label(axs[0], 'a', x=0.04, y=1.06, color='k')
axs[0].set_title('Measurement', fontsize=7, pad=3)
plot_label(axs[1], 'b', x=0.04, y=1.06, color='k')
axs[1].set_title('Simulation', fontsize=7, pad=3)
axs[0].set_xlim(enlarge(xtest)[0], enlarge(xtest)[-1])
axs[1].set_xlim(axs[0].get_xlim())
axs_top = axs[0].twiny()
axs_top.set_xlim(np.array(axs[0].get_xlim()) / Lfactor)
axs_top.set_xlabel('$Q_L$ ($2\\pi/c_L$)')
axs_top.tick_params(direction='in')
axs_top = axs[1].twiny()
axs_top.set_xlim(np.array(axs[0].get_xlim()) / Lfactor)
axs_top.tick_params(labeltop=False, direction='in')
fig.savefig('Ta_LMap_new.pdf', bbox_inches='tight', pad_inches=0.02)


## Supplementary Fig. S7 — Ta representative RIXS cuts

The three panels show the selected raw spectra, their background/self-absorption-corrected counterparts, and the corresponding simulated cuts. The legend uses the magnitude of the second HKL component, while the model retains the scan geometry.

Originally plotted with Matplotlib 3.8.0


In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    3,
    1,
    sharex=True,
    sharey=True,
    figsize=(6.75 / 2, 8.75 / 2),
    dpi=150,
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

for scan0, scan1, scan2, color in zip(
    Cmap_data_raw,
    Cmap_data,
    Cmap_calc,
    ['C1', 'C0', 'C3', 'C2'],
):
    choose1 = scan1['x'] <= 3.0
    choose2 = scan2['x'] <= 3.0
    axs[0].errorbar(
        scan0['x'][choose1],
        scan0['y'][choose1],
        scan0['yerr'][choose1],
        fmt='o',
        color=color,
        ls='--',
        lw=0.5,
        ms=1,
        label='$Q$=({:.2f}, {:.2f}, {:.2f})'.format(
            scan0['HKLc'][0],
            abs(scan0['HKLc'][1]),
            scan0['HKLc'][2],
        ),
    )
    axs[1].errorbar(
        scan1['x'][choose1],
        scan1['y'][choose1],
        scan1['yerr'][choose1],
        fmt='o',
        color=color,
        ls='--',
        lw=0.5,
        ms=1,
        label='$Q$=({:.2f}, {:.2f}, {:.2f})'.format(
            scan1['HKLc'][0],
            abs(scan1['HKLc'][1]),
            scan1['HKLc'][2],
        ),
    )
    axs[2].plot(
        scan2['x'][choose2],
        scan2['y'][choose2],
        color=color,
        ls='--',
        lw=0.5,
        ms=1,
        label='$Q$=({:.2f}, {:.2f}, {:.2f})'.format(
            scan2['HKLc'][0],
            abs(scan2['HKLc'][1]),
            scan2['HKLc'][2],
        ),
    )

axs[0].legend(
    loc='upper right',
    frameon=False,
    handletextpad=0.3,
    labelspacing=0.2,
    fontsize=6,
    title='Raw data',
)
axs[1].legend(
    loc='upper right',
    frameon=False,
    handletextpad=0.3,
    labelspacing=0.2,
    fontsize=6,
    title='Corrected data',
)
axs[2].legend(
    loc='upper right',
    frameon=False,
    handletextpad=0.3,
    labelspacing=0.2,
    fontsize=6,
    title='Simulation',
)

for ax in axs:
    make_theme(ax, ticklength=4)
    ax.axhline(y=0.0, ls='--', lw=0.5, color='k', zorder=100)

axs[1].set_xlim(-0.25, 2.0)
axs[1].set_xticks(np.arange(0.0, 2.01, 0.5))
axs[1].set_ylim(-0.08, 0.6)
axs[1].set_yticks(np.arange(0.0, 1.01, 0.4))
axs[2].set_xlabel('Energy loss (eV)')
axs[1].set_ylabel('Intensity (arb. units)')
axs[1].yaxis.set_label_coords(-0.09, 0.0)
plot_label(axs[0], 'a', x=0.04, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.04, y=0.9, color='k')
plot_label(axs[2], 'c', x=0.04, y=0.9, color='k')

fig.savefig('Ta_CutMap_new.pdf')


## Supplementary Fig. S8 — Ta momentum-dependent QFI

The left panel compares the representative Ta result, fit **9**, with the 48-member experimental/model ensemble. The right panel compares polarization channels; row **0** of each polarized file represents that same fit.

The theoretical curves are smoothed with a Savitzky–Golay filter (`window_length=11`, `polyorder=3`); the experimental ensemble points and error bars are not smoothed.

Originally plotted with Matplotlib 3.11.2

In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    1,
    2,
    dpi=150,
    sharey=True,
    figsize=(6.75 / 2, 6.75 * 2 / 9),
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

x = Ls[:][ta_momentum_indices] * Lfactor
y = QFI_mean
yerr = [QFI_std, QFI_std]
y_smooth = savgol_filter(theorSpectra[9], window_length=11, polyorder=3)
axs[0].plot(newmomenta[:], y_smooth, label='theory')
axs[0].errorbar(x, y, yerr=yerr, fmt='x', capsize=3, label='meas.+model')
axs[0].axvline(x=18.4 * Lfactor, linestyle='--', linewidth=1, color='gray')

ts_smooth = savgol_filter(theorSpectra[9], window_length=11, polyorder=3)
axs[1].plot(newmomenta, ts_smooth, label=f'unpolarized')
ts_smooth0 = savgol_filter(theorSpectraM0[0], window_length=11, polyorder=3)
axs[1].plot(newmomenta, ts_smooth0, label='in $\\pi$, out $\\pi$')
ts_smoothpi2 = savgol_filter(theorSpectraMpi2[0], window_length=11, polyorder=3)
axs[1].plot(newmomenta, ts_smoothpi2, label='in $\\pi$, out $\\sigma$')
axs[1].axvline(x=18.4 * Lfactor, linestyle='--', linewidth=1, color='gray')

axs[0].legend(fontsize=6, loc=(0.15, 0.75), frameon=False)
axs[1].legend(fontsize=6, loc='best', frameon=False)

for ax in axs:
    make_theme(ax, ticklength=4)

axs[0].set_ylim(0.5, 1.5)
axs[0].set_ylabel('nQFI')

for i in range(2):
    axs[i].set_xlim(-0.2, 3.7)
    axs[i].set_xticks(np.arange(0, 3.1, 1))
    axs[i].set_xlabel('$Q_L$ $(2\\pi/d)$')
    axs[i].axhline(y=1, color='r', linestyle='--', lw=0.5)
    axs_top = axs[i].twiny()
    axs_top.set_xlim(-0.2 / Lfactor, 3.7 / Lfactor)
    axs_top.set_xticks(np.arange(0, 18.1, 6))
    axs_top.tick_params(direction='in')
    axs_top.set_xlabel('$Q_L$ ($2\\pi/c_L$)')

plot_label(axs[0], 'a', x=0.08, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.08, y=0.9, color='k')

plt.savefig('Ta_exp_new.pdf', bbox_inches='tight', pad_inches=0.01)


## Supplementary Fig. S9 — Ta incident-energy-dependent QFI

The experimental/model points and error bars summarize the 24-member ensemble using incident-energy normalization only. The unpolarized theory uses Ta fit **9**, and polarized arrays use their corresponding single-fit row **0**.

The left theoretical curve is Savitzky–Golay smoothed (`window_length=11`, `polyorder=3`), while the right unpolarized curve is plotted without smoothing. The displayed nQFI range is `0.8–1.0`.

Originally plotted with Matlplotlib 3.11.2

In [ ]:
paper_rc_qfi9()

# Figure-layout state verified against the original production notebook.
fig, axs = plt.subplots(
    1,
    2,
    dpi=150,
    sharey=True,
    figsize=(6.75 / 2, 6.75 * 2 / 9),
    gridspec_kw=dict(left=0.12, right=0.96, bottom=0.1, top=0.96, hspace=0.0),
)

x = omega_data
y = QFI_meanE
yerr = [QFI_stdE, QFI_stdE]
y_smooth = savgol_filter(theorSpectra0[9], window_length=11, polyorder=3)
axs[0].plot(omega11, y_smooth, label='theory')
axs[0].errorbar(x, y, yerr=yerr, fmt='*', capsize=3, label='meas.+model')
axs[0].axvline(x=11.2155, linestyle='--', linewidth=1, color='gray')

axs[1].plot(omega11, theorSpectra0[9], label='unpolarized')
axs[1].plot(omega11, theorSpectraE0[0], label='in $\\pi$, out $\\pi$')
axs[1].plot(omega11, theorSpectraEpi2[0], label='in $\\pi$, out $\\sigma$')
axs[1].axvline(x=11.2155, linestyle='--', linewidth=1, color='gray')

axs[0].legend(loc='best', fontsize=6, frameon=False)
axs[1].legend(loc='best', fontsize=6, frameon=False)

for ax in axs:
    make_theme(ax, ticklength=4)

axs[0].set_ylim(0.8, 1.0)
axs[0].set_yticks(np.arange(0.8, 1.0, 0.05))
axs[1].set_ylim(0.8, 1.0)
axs[0].set_xlabel('Incident energy (keV)')
axs[1].set_xlabel('Incident energy (keV)')
axs[0].set_ylabel('nQFI')
axs[0].axhline(y=1, color='r', linestyle='--', lw=0.5)
axs[1].axhline(y=1, color='r', linestyle='--', lw=0.5)
plot_label(axs[0], 'a', x=0.08, y=0.9, color='k')
plot_label(axs[1], 'b', x=0.08, y=0.9, color='k')
plt.savefig('Ta_simulation_new.pdf', bbox_inches='tight', pad_inches=0.01)
